# 260312 OpenAI API 실습 - Chat Completions & 파라미터 튜닝

**Day 3 · 2026-03-12 (목)** · 1주차 W1 · API & Chain 기초

---

## 오늘의 학습 목표
- `openai` SDK로 `chat.completions.create()` 호출하기
- 메시지 `role`(system / user / assistant)로 페르소나 + 대화 맥락 주입
- 생성 파라미터(`temperature`, `top_p`, `max_tokens`, `frequency_penalty`, `presence_penalty`, `stop`, `seed`) 감 잡기
- 멀티턴 대화(히스토리 append 패턴) 직접 만들어 보기
- `stream=True`로 청크 단위 응답 받기

> 비유: 오늘은 **자동차 계기판을 하나씩 만져보는 날**. 핸들(role)·액셀(temperature)·제한속도(max_tokens)·브레이크(stop)를 따로따로 밟아보며 GPT가 어떻게 반응하는지 몸으로 익힌다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w1_api_and_chain/llm_260312_openAi_practice.ipynb)

## 0. 환경 준비

OpenAI 공식 SDK와 `python-dotenv` 설치. Colab에서는 `userdata.get()`, 로컬에서는 `.env` 파일을 사용한다.

> 비유: API 키는 **현관 열쇠**. 노트북 안에 박아두면 GitHub에 올렸을 때 도둑이 퍼간다. 그래서 `.env` 또는 Colab Secret으로 분리 보관한다.

In [ ]:
# openai SDK + dotenv 설치 (Colab/로컬 공통)
!pip install -q openai python-dotenv

In [ ]:
# API 키 로드: Colab 우선, 실패하면 .env 사용
import os

api_key = None
try:
    # Colab: 좌측 열쇠 아이콘 > OPENAI_API_KEY 등록 후 노트북 액세스 켜기
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    # 로컬: 동일 폴더(또는 상위) .env 파일의 OPENAI_API_KEY=... 줄을 읽음
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv('OPENAI_API_KEY')

# 키가 제대로 로드됐는지 앞 8글자만 확인 (전체를 찍지 말 것)
assert api_key, 'OPENAI_API_KEY 가 로드되지 않았어요. Secret 또는 .env 확인!'
print(api_key[:8])

## 1. OpenAI 클라이언트 초기화

`OpenAI()` 객체는 앞으로 모든 API 호출의 창구. `client.chat.completions.create(...)` 가 핵심 진입점이다.

> 비유: `client`는 **콜센터 전용 수화기**. 매번 수화기를 새로 들지 않고 한 번 들어둔 뒤 계속 통화한다.

In [ ]:
# OpenAI 클라이언트 객체 생성 - 이후 모든 호출은 이 client를 통해서
from openai import OpenAI

client = OpenAI(api_key=api_key)
type(client)  # openai.OpenAI 클래스 확인

## 2. 첫 API 호출 - `chat.completions.create`

필수 인자 2개: `model` (어떤 모델) + `messages` (리스트 형태의 대화). 강의에서는 비용이 저렴한 `gpt-4o-mini`로 실습한다.

응답 객체(`ChatCompletion`)는 트리 구조:
- `response.choices[0].message.content` → 실제 답변 텍스트
- `response.usage.total_tokens` → 입력+출력 합산 토큰
- `response.choices[0].finish_reason` → 'stop' / 'length' / 'content_filter' 등

> 비유: 응답은 **택배 박스**. 겉에는 운송장(id, model, usage)이 붙어 있고, 박스를 열면 실제 물건(`message.content`)이 들어 있다.

In [ ]:
# 질문 3개를 연속 호출 - 각각 새로운 대화로 처리됨 (세션 공유 X)
questions = [
    'python이 무엇인지 한 문장으로 설명해 주세요.',
    'RAG이 무엇인지 한 문장으로 설명해 주세요.',
    'LLM이 무엇인지 한 문장으로 설명해 주세요.',
]

for q in questions:
    response = client.chat.completions.create(
        model='gpt-4o-mini',  # 저렴하고 한국어도 OK
        messages=[{'role': 'user', 'content': q}],  # role=user: 사람이 묻는 질문
    )
    # choices는 생성된 답변들의 리스트, 보통 첫 번째만 사용
    print(response.choices[0].message.content)
    print('---')

In [ ]:
# 응답 객체 전체를 한 번 살펴보자 - 운송장 정보가 전부 보인다
print(response)

In [ ]:
# 자주 쓰는 메타 정보 3종 세트
print('id   :', response.id)            # 이 요청의 고유 ID (트러블슈팅 시 유용)
print('model:', response.model)         # 실제 응답한 모델 버전
print('stop :', response.choices[0].finish_reason)  # 왜 답변이 끝났는지
print('usage:', response.usage.total_tokens, 'tokens')

## 3. Role - system / user / assistant

- **system**: 개발자가 정하는 **페르소나·규칙**. "당신은 OO입니다" 류.
- **user**: 실제 사용자가 입력한 질문.
- **assistant**: 모델이 **직전에 답한 내용**. 대화 기록을 다시 모델에게 넘길 때 사용.

> 비유: 영화 촬영장. **system = 감독의 연기 지시서**, **user = 상대 배우의 대사**, **assistant = 내가 방금 친 대사**. 세 가지가 모여야 맥락 있는 연기가 나온다.

강사 팁: 시스템 프롬프트가 길어져도 저렴한 모델은 100% 따르지 않을 수 있다. 규칙이 복잡해질수록 모델 성능·프롬프트 구조화가 함께 따라와야 한다.

In [ ]:
# system 메시지로 페르소나 주입 - 같은 질문도 말투·관점이 달라진다
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': '당신은 한국의 역사 선생님입니다.'},  # 페르소나
        {'role': 'user',   'content': '대한민국의 수도는?'},
    ],
)
response.choices[0].message.content

In [ ]:
# 페르소나 실험 1 - 친절한 A/S 상담원 (이모지 적극 활용)
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': '당신은 A/S 상담원입니다. 존댓말을 사용하고 이모지를 최대한 활용합니다.'},
        {'role': 'user',   'content': '비밀번호를 잊어버렸어요'},
    ],
)
response.choices[0].message.content[:150]

In [ ]:
# 페르소나 실험 2 - 초등학교 선생님 (쉽고 간결한 설명)
response1 = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': '당신은 초등학교 선생님입니다. 최대한 쉽게 간결하게 설명해 주세요.'},
        {'role': 'user',   'content': '우주는 왜 신비로운가요?'},
    ],
)
print(response1.choices[0].message.content)

In [ ]:
# 페르소나 실험 3 - 근현대사 박사 (정확한 사실관계만)
response2 = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': '당신은 근현대사 공부를 많이 한 박사입니다. 정확한 사실관계만 전달합니다.'},
        {'role': 'user',   'content': '1994년에 일어났던 가장 끔찍한 사건은?'},
    ],
)
print(response2.choices[0].message.content)

## 4. 토큰(Token)이란?

모델이 글자를 그대로 먹는 게 아니라 **BPE(Byte-Pair Encoding)** 로 쪼개서 숫자 ID로 바꾼 뒤 처리한다. 대충:
- 한글: 한 단어 ≈ **2 토큰**
- 영어: 한 단어 ≈ **1 ~ 1.5 토큰**
- `apples` → `apple` + `s`, `studied` → `stud` + `ied` 식으로 서브워드로 분해된다.

남은 잔여 토큰은 API로 조회 불가. 과금은 `prompt_tokens + completion_tokens` 기준.

> 비유: 토큰은 **택시 미터기의 기본구간**. 내가 탄 거리(문장)가 몇 개 구간(토큰)인지는 타봐야 확정된다.

## 5. `temperature` - 창의성의 불 조절

- **0에 가까울수록** 결정적(같은 입력 → 같은 답). FAQ·고객응대에 적합.
- **높을수록** 다음 토큰 확률 분포가 평탄해져 다양한 단어가 샘플링된다. 시·아이디어 브레인스토밍에 적합.
- GPT-5 계열은 reasoning 모델이라 `temperature`가 1로 고정되어 변경 불가.

> 비유: **엑셀 페달의 깊이**. 살짝 밟으면(0) 정해진 코스만, 꾹 밟으면(1.5) 차선을 바꿔가며 달린다.

In [ ]:
# temperature=0 : 3번 호출해도 거의 동일한 답변
question = '인공지능의 미래에 대해 한 문장으로 말해봐!'
for i in range(3):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': question}],
        temperature=0,  # 결정적
    )
    print(f'{i+1}: {response.choices[0].message.content}')

In [ ]:
# temperature=1.5 : 매번 다른 표현이 튀어나옴
for i in range(3):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': question}],
        temperature=1.5,  # 창의적
    )
    print(f'{i+1}: {response.choices[0].message.content}')

In [ ]:
# 여러 temperature 값으로 2번씩 - 값이 커질수록 답변 다양성 증가 확인
question = '고양이의 특성에 대해 한 문장으로 알려줘'
for temp in [0, 0.5, 1.0]:
    print(f'=== temperature={temp} ===')
    for i in range(2):
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': question}],
            temperature=temp,
        )
        print(f'  {i+1}: {response.choices[0].message.content}')

### 5-1. (심화) temperature가 softmax에 미치는 영향

강의 중 소개된 식: `q_i = exp(z_i / T) / Σ exp(z_j / T)`.
T(temperature)가 작을수록 분포가 뾰족해져 1등 토큰만 뽑히고, T가 크면 평탄해져 다 고만고만해진다.

In [ ]:
# softmax를 직접 구현해 temperature에 따른 확률 분포 변화 확인
import numpy as np

def calculate_softmax(logits, T):
    z = np.array(logits) / T           # 온도로 나누기
    z = z - np.max(z)                  # 수치 안정화 (큰 값 빼기)
    e_z = np.exp(z)
    return e_z / e_z.sum()

logits = [2.0, 1.5, 1.0, 0.5]
for T in [0.1, 0.5, 1.0, 2.0, 10.0, 100.0]:
    probs = calculate_softmax(logits, T)
    print(f'T={T:>6}  →  {np.round(probs, 4)}')

## 6. `max_tokens` - 답변 길이 상한

답이 길어 비용이 폭주하는 걸 막기 위해 상한을 건다. 한도를 넘기면 `finish_reason='length'`로 잘린 채 반환.

> 비유: **편지지 한도**. 편지지를 3장만 주면, 아무리 하고 싶은 말이 많아도 3장 끝나는 순간 펜을 내려놓는다.

In [ ]:
# max_tokens=50으로 강제 제한 - finish_reason이 'length'가 될 것
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': '대한민국에 대해 알려줘'}],
    max_tokens=50,
)
print(response.choices[0].message.content)
print('---')
print('finish_reason:', response.choices[0].finish_reason)

In [ ]:
# temperature × max_tokens 조합 실험 - 파라미터 감각 익히기
temps = [0, 0.5, 1.0]
max_tokens_list = [50, 100, 200]
question = '독일의 초등학교 교육 시스템에 대해서 설명해 주세요.'

for t in temps:
    for m in max_tokens_list:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': question}],
            temperature=t,
            max_tokens=m,
        )
        print(f'[T={t}, max={m}]')
        print(response.choices[0].message.content)
        print('-' * 30)

## 7. `top_p` - 누적 확률 컷 (nucleus sampling)

상위 토큰들의 **누적 확률 합이 p가 될 때까지만** 샘플링 후보로 둔다. `temperature`와 역할이 비슷하므로 **둘 중 하나만** 바꾸는 게 정석.

> 비유: 반에서 성적 상위 **몇 %까지** 추첨 후보로 올릴지 정하는 것. 10%(0.1)면 상위권만, 90%(0.9)면 거의 전체가 후보.

In [ ]:
# top_p=0.1 : 상위 10% 단어만 사용 - 안전/일관된 답변
question = '겨울을 주제로 짧은 시를 써 줘.'
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    top_p=0.1,
)
print(response.choices[0].message.content)

In [ ]:
# top_p=1.0 : 제한 없음 - 더 다양한 어휘 선택 가능
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    top_p=1,
)
print(response.choices[0].message.content)

## 8. `frequency_penalty` & `presence_penalty`

- **frequency_penalty** (0 ~ 2): 같은 **단어**가 자주 나올수록 감점. 반복 방지.
- **presence_penalty** (0 ~ 2): 이미 등장한 **단어/주제**면 감점. 새로운 주제 유도.

옛날 싸구려 모델들이 `\n\n\n...` 또는 특수문자만 뱉는 사고를 방지할 때 유용했다. 현재 `gpt-4o-mini`는 기본값이면 대체로 괜찮다.

> 비유: **frequency_penalty = 같은 노래 재생 금지**, **presence_penalty = 같은 가수 앨범 금지**. 전자는 중복 단어, 후자는 중복 주제를 막는다.

In [ ]:
# frequency_penalty=0 (기본) : 반복 허용
question = '과학에 대해 세 문장으로 설명해 줘'
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    frequency_penalty=0,
)
print(response.choices[0].message.content)

In [ ]:
# frequency_penalty=1.5 : 같은 단어 반복 강하게 억제
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    frequency_penalty=1.5,
)
print(response.choices[0].message.content)

In [ ]:
# presence_penalty=0 : 같은 주제 허용
question = '과일에 대해 세 문장으로 설명해 줘'
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    presence_penalty=0,
)
print(response.choices[0].message.content)

In [ ]:
# presence_penalty=1.5 : 새로운 단어/주제로 전환 유도
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    presence_penalty=1.5,
)
print(response.choices[0].message.content)

## 9. `stop` & `seed` - 종료 제어 & 재현성

- **stop**: 지정한 문자열이 생성되면 **그 직전까지만** 출력하고 즉시 종료. 번호 목록을 '6.' 앞에서 끊는 식으로 활용.
- **seed**: 같은 입력+같은 seed면 **유사한 결과**를 재현(100% 보장은 아님). 실험/디버깅용.

> 비유: **stop = 자동 브레이크**(특정 단어가 보이면 급정거), **seed = 녹음된 카세트 테이프**(대충 같은 멜로디가 흘러나옴).

In [ ]:
# stop 파라미터 - '6. '이 생성되는 순간 바로 종료 → 1~5번만 출력
question = '프로그래밍 언어 10개를 나열해 줘'
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    stop=['6. '],
)
print(response.choices[0].message.content)
print('finish_reason:', response.choices[0].finish_reason)

In [ ]:
# stop 없을 때 - 10개 풀로 출력
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
)
print(response.choices[0].message.content)

In [ ]:
# seed 동일 - 두 호출 결과가 거의 유사해야 함
question = '바나나에 대해서 설명해 줘'
for _ in range(2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': question}],
        seed=56,  # 같은 seed = 비슷한 결과
    )
    print(response.choices[0].message.content[:120])
    print('---')

In [ ]:
# stop + seed 조합 - 재현성 있는 제어된 출력
question = '열대 과일 10개를 나열해 줘'

# 바나나 등장 전까지만
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    seed=56,
    stop=['바나나'],
)
print('[stop=바나나]')
print(response.choices[0].message.content)

# 파인애플 등장 전까지만
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
    seed=56,
    stop=['파인애플'],
)
print('[stop=파인애플]')
print(response.choices[0].message.content)

## 10. 멀티턴 대화 - AI에게 '기억'을 주는 법

`client.chat.completions.create()` 는 **매 호출마다 새 통화**다. 이전 대화를 기억시키려면 내가 직접 `messages` 리스트에 히스토리를 **차곡차곡 append** 해서 통째로 다시 넘겨야 한다.

패턴:
1. system 메시지로 페르소나 1회 세팅
2. user 질문 append → 호출 → 응답(assistant) 받기
3. **받은 assistant 메시지도 리스트에 append**
4. 다음 user 질문을 또 append → 호출 → ... 반복

> 비유: 전화는 끊었다 걸 때마다 상대가 **기억상실**이라, 매번 **지난 대화록 복사본**을 같이 들려줘야 이어서 말해준다. 대신 기록이 쌓일수록 토큰 비용도 함께 쌓임.

In [ ]:
# 히스토리 누적 - 처음엔 간단하게 append로
messages = [
    {'role': 'system', 'content': '당신은 IT 헬프데스크입니다. 간결하게 답변해 주세요'},
]
messages.append({'role': 'user', 'content': '비밀번호를 잊어버렸어요'})

response = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
answer1 = response.choices[0].message.content
print('assistant:', answer1)

# 중요: 모델의 답변을 다시 리스트에 넣어줘야 맥락이 이어진다
messages.append({'role': 'assistant', 'content': answer1})

In [ ]:
# 두 번째 턴 - 맥락이 살아있는지 확인
messages.append({'role': 'user', 'content': '그 방법이 안되면 어떡하죠?'})

response = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
answer2 = response.choices[0].message.content
messages.append({'role': 'assistant', 'content': answer2})
print('assistant:', answer2)

In [ ]:
# 세 번째 턴 - '처음에 뭘 물어봤는지' 제대로 기억하는지 테스트
messages.append({'role': 'user', 'content': '내가 처음에 뭘 물어봤지?'})

response = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
answer3 = response.choices[0].message.content
print('assistant:', answer3)
print('대화 길이:', len(messages), '개 메시지')
print('누적 토큰:', response.usage.total_tokens)

In [ ]:
# 실전 패턴 - 멀티턴을 함수로 감싸기 (깔끔하게 재사용)
messages = [
    {'role': 'system', 'content': '당신은 의사입니다. 환자를 진단하고 있어요. 친절하고 간결하게 설명해 주세요'},
]

def send_message_and_print(user_content):
    """유저 발화를 누적하고 답변을 받아 assistant 기록까지 append."""
    messages.append({'role': 'user', 'content': user_content})
    print(f'**환자:** {user_content}')

    response = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
    assistant_content = response.choices[0].message.content
    messages.append({'role': 'assistant', 'content': assistant_content})
    print(f'**의사:** {assistant_content}\n')

# 3턴 시나리오 실행
send_message_and_print('머리가 아파요')
send_message_and_print('수면 부족인 것 같은데, 그럼 배도 아픈가요? 어떻게 하면 좋아질까요?')
send_message_and_print('자기전에 마사지도 도움이 되나요?')

## 11. `stream=True` - 실시간 청크 출력

기본 호출은 **모델이 답 완성할 때까지 기다렸다가** 한 번에 받는다. 긴 응답은 답답하다. `stream=True`를 주면 답변이 생성되는 대로 **청크(chunk)** 단위로 즉시 전달된다.

> 비유: 택배를 **완성 후 한꺼번에 배달**(일반)하느냐, **만드는 대로 조각 조각 배달**(스트리밍)하느냐의 차이. 사용자는 기다림이 줄어들고, 개발자는 UX가 살아난다.

주의: chunk의 의미는 BPE 토큰이 **아니다**. 내부 전송 단위일 뿐이며 모델마다 크기가 다르다.

In [ ]:
# stream=True - 한 글자(사실은 청크)씩 실시간 출력
messages = [{'role': 'user', 'content': '파이썬의 장점 세 가지를 얘기해 줘'}]

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    stream=True,  # 제너레이터 형태로 리턴
)

full_answer = ''
for chunk in response:
    text = chunk.choices[0].delta.content  # delta.content에 이번 조각이 들어 있음
    if text:
        print(text, flush=True, end='')   # 줄바꿈 없이 이어 붙이기
        full_answer += text

print('\n---')
print('전체 길이:', len(full_answer), '자')

## 정리 - 오늘 체크리스트

- [x] `client.chat.completions.create()` 한 번은 직접 찍어봤다
- [x] role 3종 (system / user / assistant)을 섞어 페르소나를 줬다
- [x] `temperature`, `max_tokens`, `top_p`, penalty, `stop`, `seed` 의 감을 익혔다
- [x] 멀티턴 대화에서 **assistant 응답도 messages에 다시 append** 한다는 걸 체득했다
- [x] `stream=True` 로 청크 수신을 경험했다

다음 시간(Day 4): 같은 일을 **LangChain**으로 더 간결하게 짜보기. Prompt Template + LCEL 파이프(`|`) 입문.